This part of the pipeline processes the raw output generated by the microbeAnnotator tool and generates a clustermap of the KEGG module completenesses.

### Paths and parameters

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as ppt
import seaborn as sns
import os
import pandas as pd
from colorcet import glasbey
from scipy.cluster import hierarchy

#### Pipeline input folders

In [ ]:
output_folder = "07-KEGGCompleteness/output"
grouping_table_file = "02-GTDB/filtered_classification_table"

#### Pipeline output folders

In [ ]:
task_root = "07-KEGGCompleteness"
results_folder = "07-KEGGCompleteness/processed_output"

In [ ]:
os.makedirs(results_folder, exist_ok=True)

#### Tool pointers and parameters

In [ ]:
inclusion_threshold = 50

In [ ]:
custom_palette = sns.husl_palette()
custom_palette = [custom_palette[0], custom_palette[1], custom_palette[2], custom_palette[4]]

### Processing results

#### Parsing output

In [ ]:
completeness = pd.read_table(output_folder + '/metabolic_summary__module_completeness.tab')

In [ ]:
# Filtering out KEGG modules that have a maximum completeness less than 50%.
completeness.columns = list(map(lambda x: '.'.join(x.split('.')[:2]), list(completeness.columns)))
completeness['maximum_presence'] = completeness.max(axis = 1, numeric_only = True)
completeness_toplot = completeness[completeness['maximum_presence'] >= inclusion_threshold].drop(columns = ['maximum_presence'])
completeness_toplot

#### Load cluster annotations

In [ ]:
grouping_table = pd.read_table(grouping_table_file, sep = '\t', header = None, names = ['accession', 'group'])
grouping_table = grouping_table.set_index('accession').squeeze()

In [ ]:
grouping_table

In [ ]:
group_legend = dict(zip(grouping_table.unique(), custom_palette))
group_legend

In [ ]:
sns.husl_palette()

In [ ]:
test = sns.husl_palette()
test

In [ ]:
grouping_table_with_colours = grouping_table.transform(lambda x: group_legend[x])
grouping_table_with_colours.name = "GTDB order"
grouping_table_with_colours

#### Defining colour groups for KEGG pathway modules

In [ ]:
## Make a pathway to colour mapping via the pathway group
# get all pathway groups
pathway_groups = list(completeness_toplot['pathway group'].unique())

# mapping pathway groups to colours
palette = sns.color_palette(glasbey, n_colors = len(pathway_groups), as_cmap = True) 
pathway_colours = dict(zip(pathway_groups, palette))

# mapping pathways to pathway groups
name_pathway = dict(zip(*completeness_toplot[['name', 'pathway group']].to_dict(orient = 'list').values()))

# connecting the two mappings for all pathways
group_colours = {name: pathway_colours[name_pathway[name]] for name in completeness_toplot['name']}

#### Clustering the completenesses

In [ ]:
completeness_numeric = completeness_toplot.select_dtypes(include = "number")
linkage = hierarchy.linkage(completeness_numeric.T, method = 'ward', metric = 'euclidean', 
                            optimal_ordering = True) # with optimal ordering for visualising any lower-level pattern

In [ ]:
row_linkage = hierarchy.linkage(completeness_numeric, method = 'ward', metric = 'euclidean', 
                            optimal_ordering = True)

#### Clustermap

In [ ]:
h = sns.clustermap(completeness_numeric, col_linkage = linkage, col_cluster = True, row_linkage = row_linkage, row_cluster = True,
                   xticklabels = False, yticklabels = True,
                   figsize = (9,20), dendrogram_ratio = (0.2, 0.05),
                   col_colors = grouping_table_with_colours, cmap = "magma_r")
h.ax_row_dendrogram.set_visible(False)
h.ax_heatmap.set_yticklabels(completeness_toplot['name'].iloc[h.dendrogram_row.reordered_ind])
h.ax_heatmap.set_xlabel('Genome assemblies', size = 18)
h.ax_heatmap.set_ylabel('KEGG module', size = 18)
h.ax_heatmap.collections[0].colorbar.set_label('Module completeness (%)', size = 16)

# The legend has to be composed on the fly. Otherwise, we can't use our pathway group-level label colouring
legend_patches = []
pathways_covered = []
for label in h.ax_heatmap.get_yticklabels():
    text = label.get_text()
    pathway = name_pathway[text]
    color = group_colours[text]
    # change the label of the pathway module at the tick label to the appropriate colour
    label.set_color(color)
    # only add a new entry to the legend if we haven't encountered a pathway from this group yet
    if pathway not in pathways_covered:
        legend_patches.append(ppt.Patch(color = color, label = pathway))
        pathways_covered.append(pathway)
        
plt.legend(handles = legend_patches, ncol = 1, bbox_to_anchor = (42,1,1,0))
# plt.savefig(os.path.join(results_folder, 'module_completeness.svg'))
plt.show()

#### Clustermap without pathway labels

In [ ]:
h = sns.clustermap(completeness_numeric, col_linkage = linkage, col_cluster = True, row_linkage = row_linkage, row_cluster = True, 
                   xticklabels = False, yticklabels = False, cbar_pos = (0.01,0.5,0.05,0.4),
                   figsize = (6,5), dendrogram_ratio = (0.2, 0.05),
                   col_colors = grouping_table_with_colours, cmap = 'magma_r')
h.ax_row_dendrogram.set_visible(False)
h.ax_heatmap.set_xlabel('Genome assemblies', size = 12)
h.ax_heatmap.collections[0].colorbar.set_label('Module completeness (%)', size = 11)

# plt.savefig(os.path.join(results_folder, 'module_completeness_noTicks.svg'))
plt.show()

#### Extracting the versatile and non-versatile IDs

In [ ]:
# left one is the versatile branch
linkage_tree = hierarchy.to_tree(linkage)
versatile_ids = completeness_numeric.columns[linkage_tree.left.pre_order(lambda x: x.id)].to_frame()
non_versatile_ids = completeness_numeric.columns[linkage_tree.right.pre_order(lambda x: x.id)].to_frame()

In [ ]:
versatile_ids.to_csv(os.path.join(results_folder, 'versatile'), index = False, header = False)
non_versatile_ids.to_csv(os.path.join(results_folder, 'non_versatile'), index = False, header = False)

In [ ]:
len(versatile_ids)

#### Plotting the versatility fraction per group

In [ ]:
versatile_frac = grouping_table[grouping_table.index.isin(versatile_ids.index)].value_counts() / grouping_table.value_counts()
versatile_frac = versatile_frac.to_frame().reset_index()

In [ ]:
plt.figure(figsize = (4,4))
ax = sns.barplot(data = versatile_frac, y = 'group', x = 'count', orient = 'h', palette = group_legend, width = 0.6)
ax.set_xlabel("Versatile fraction (%)")
ax.set_ylabel("GTDB order")
fig = ax.get_figure()
fig.savefig(os.path.join(results_folder, 'versatility_fractions.svg'))
plt.show()